# 🔢 Cuaderno 01 — Álgebra Lineal

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Este cuaderno es la **puerta de entrada** a la Unidad 3. Aprenderás a manipular matrices y
vectores con una sintaxis **estilo MATLAB** que elimina la fricción de Python.

### 🎯 Objetivos
- Construir matrices y vectores con `M[[...]]`, `I[n]`, `zeros`, `ones`, `eye`.
- Hacer operaciones de renglón legibles para la **eliminación Gaussiana**.
- Formar la **matriz aumentada** con `c_[A, b]` y concatenar con `r_`.
- Calcular determinante, inversa y resolver sistemas con `solve`.
- Introducir **álgebra simbólica** con `var`, `jacobiano` y `mostrar`.


In [1]:
#@title ⚙️ Cargar entorno estilo MATLAB (ejecuta PRIMERO)

import os, sys, urllib.request

# --- Detecta si corremos en Google Colab ---
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

ARCHIVO = "robomat.py"
URL = "https://raw.githubusercontent.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/main/lib/robomat.py"


def _buscar_robomat(nombre=ARCHIVO, saltos=6):
    """Busca robomat.py junto al notebook, en la carpeta actual y en lib/ de los padres."""
    bases = [os.getcwd()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    if ruta_nb:
        bases.append(os.path.dirname(os.path.abspath(ruta_nb)))
    for base in list(bases):
        b = base
        for _ in range(saltos):
            for cand in (os.path.join(b, nombre), os.path.join(b, "lib", nombre)):
                if os.path.exists(cand):
                    return os.path.abspath(cand)
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


ruta_robomat = _buscar_robomat()
if ruta_robomat is None:
    print("robomat.py no está en disco; descargando del repositorio...")
    urllib.request.urlretrieve(URL, ARCHIVO)
    ruta_robomat = os.path.abspath(ARCHIVO)

_carpeta_lib = os.path.dirname(ruta_robomat)
if _carpeta_lib not in sys.path:
    sys.path.insert(0, _carpeta_lib)

# Import real (no exec): Pylance lo resuelve y da autocompletado.
import importlib
import robomat as _robomat
importlib.reload(_robomat)          # recoge los cambios si editaste la libreria
from robomat import *               # noqa: F401,F403

print(f"✅ Entorno cargado desde: {ruta_robomat}")


✅ Entorno cargado desde: o:\Mi unidad\CursoMN\U3 Sistemas de ecuaciones lineales algebráicas\U3_Repo\lib\robomat.py


## 1. Construir matrices y vectores

Un sistema $\mathbf{A}\mathbf{x}=\mathbf{b}$ se arma con una **matriz de coeficientes**
$\mathbf{A}$ y un **vector de términos independientes** $\mathbf{b}$.

### 1.1 La forma tradicional: NumPy y SymPy

Normalmente esto se hace con dos bibliotecas, según lo que necesites:

| Biblioteca | Para qué sirve | Cómo se crea una matriz |
|---|---|---|
| **NumPy** | cálculo **numérico**: los datos son números y buscas rapidez | `np.array([[1, 2], [3, 4]])` |
| **SymPy** | cálculo **simbólico**: quieres manipular letras, derivar, simplificar | `sym.Matrix([[p, q], [r, s]])` |

Cada una tiene su propia forma de crear las cosas y sus propios detalles finos.
Por ejemplo, en NumPy un vector no es ni fila ni columna hasta que se le da forma
con dobles corchetes, y mezclar un arreglo de NumPy con una matriz de SymPy
produce errores difíciles de leer:

In [2]:
import numpy as np
import sympy as sym

A = np.array([[1, 2], 
              [3, 4]])       # NumPy: matriz numérica

p, q, r, s = sym.symbols('p q r s')
As = sym.Matrix([[p, q], 
                 [r, s]])    # SymPy: matriz simbólica

AxA = A * A        # NumPy: multiplica ELEMENTO A ELEMENTO  -> [[1, 4], [9, 16]]
print("A * A =\n", AxA)

AsxAs = As * As      # SymPy: PRODUCTO MATRICIAL              -> resultado distinto
print("As * As =\n", AsxAs)

AA = A @ A        # en ambas: producto matricial            -> [[7, 10], [15, 22]]
print("A @ A =\n", AA)

A * A =
 [[ 1  4]
 [ 9 16]]
As * As =
 Matrix([[p**2 + q*r, p*q + q*s], [p*r + r*s, q*r + s**2]])
A @ A =
 [[ 7 10]
 [15 22]]


In [3]:
np.linalg.solve(As, [1, 2])   # ✗ falla: NumPy exige números, no símbolos

UFuncTypeError: Cannot cast ufunc 'solve1' input 0 from dtype('O') to dtype('float64') with casting rule 'same_kind'

Hay dos trampas clásicas de esas bibliotecas (la biblioteca del curso ya las
resuelve; se explica en la sección 2):

1. **`*` no significa lo mismo.** En NumPy multiplica elemento a elemento; en
   SymPy es el producto matricial. El resultado es distinto sin que haya ningún
   error que te avise.
2. **Las rutinas de NumPy no aceptan símbolos.** `np.linalg.solve` o
   `np.linalg.det` fallan con un mensaje poco claro si la matriz es simbólica.

> ✅ Regla: para el **producto matricial** usa siempre `@`. Es el único operador
> que significa lo mismo en NumPy y en SymPy.

### 1.2 La forma en este curso

La biblioteca del curso (`lib/robomat.py`) **ya carga** NumPy, SymPy y Matplotlib,
y añade una sintaxis más corta: `M[[...]]` construye la matriz y **decide sola** el
tipo, mirando si hay símbolos o solo números.

| Lo que escribes | Lo que obtienes |
|---|---|
| `M[[1, 2], [3, 4]]` | arreglo de **NumPy** (todos los valores son números) |
| `M[[p, q], [r, s]]` | matriz **simbólica** de SymPy (hay letras) |

Así el mismo código sirve para los dos mundos, sin importar nada ni recordar en
cuál de las dos bibliotecas estás.

#### La forma la deciden los corchetes (como en MATLAB)

| Escribes | Obtienes | Forma |
|---|---|---|
| `M[[1, 2, 3]]` | un **renglón** | `(1, 3)` |
| `M[[1], [2], [3]]` | una **columna** | `(3, 1)` |
| `M[[1, 2], [3, 4]]` | matriz 2×2 | `(2, 2)` |

La regla: si dentro hay **números sueltos**, es un renglón; si hay **listas**,
cada lista interior es un renglón. Es la misma idea que en MATLAB, donde
`[1 2 3]` es un renglón.

In [ ]:
# --- Forma tradicional con NumPy ---
import numpy as np

A_np = np.array([[1, 2], 
                 [3, 4]])

b_np = np.array([[5], [6]])      # dos corchetes => vector COLUMNA (forma 2x1)

b_np = np.array([[5], 
                 [6]])      # Más fácil de escribir y leer, pero lo mismo que arriba

print("A_np = \n", A_np)
print("tipo:", type(A_np).__name__)
print("forma de b_np:", b_np.shape, "-> (2, 1) significa columna")
print("si se escribe np.array([5, 6]) la forma es", np.array([5, 6]).shape, "-> fila (1D)")

A_np = 
 [[1 2]
 [3 4]]
tipo: ndarray
forma de b_np: (2, 1) -> (2, 1) significa columna
si se escribe np.array([5, 6]) la forma es (2,) -> fila (1D)


In [ ]:
# --- Forma tradicional con SymPy ---
import sympy as sym

p, q, r, s = sym.symbols('p q r s')     # hay que declarar cada letra
A_sim = sym.Matrix([[p, q], [r, s]])

print("tipo:", type(A_sim).__name__)
mostrar(A_sim)                           
# mostrar() es de la biblioteca del curso para imprimir matrices bonitas

tipo: MutableDenseMatrix


<IPython.core.display.Math object>

In [ ]:
# --- Forma del curso: M[[...]] decide el tipo solo ---
A_num = M[[1, 2],
          [3, 4]]

x1, x2 = var('x1', 'x2')    # declara variables simbólicas
A_simb = M[[x1, 2],
           [3, x2]]

print("M[[1, 2],\n  [3, 4]]   ->", type(A_num).__name__)
print("M[[x1, 2],\n  [3, x2]] ->", type(A_simb).__name__)
print()
print("A_num =\n", A_num)
print("\nA_simb =")
mostrar(A_simb)

# Es exactamente lo mismo que se obtendría a mano:
print("\n¿igual a np.array([[1, 2], [3, 4]])?", np.array_equal(A_num, A_np))

M[[1, 2],
  [3, 4]]   -> ndarray
M[[x1, 2],
  [3, x2]] -> MatrizRobomat

A_num =
 [[1. 2.]
 [3. 4.]]

A_simb =


<IPython.core.display.Math object>


¿igual a np.array([[1, 2], [3, 4]])? True


In [ ]:
# Constructores rápidos, con su equivalente en NumPy al lado
#
#   zeros(n) / ones(n)       -> MATRIZ n x n        (como MATLAB)
#   zeros(m, n) / ones(m, n) -> MATRIZ m x n        (como MATLAB)
#   zeros(n, 1) / zeros(1, n)-> vector columna/fila (explicito)
#   I[n]    -> identidad n x n            (cuadrada)
#   I[m, n] -> identidad rectangular m x n
#   zeros.sym / ones.sym / eye.sym      -> las mismas, pero SIMBOLICAS
#   I.sym[n] / I.sym[m, n]               -> identidad simbolica
#   var(A)                               -> convierte A a simbolica
#  (tambien existen zeros_sym(), ones_sym(), eye_sym() como forma larga)
#
# Para una diagonal desplazada (raro) usa eye(m, n, k); I[] acepta una o dos
# dimensiones a propósito, para que no se vuelva ambiguo de leer.
Z  = zeros(3)      # np.zeros((3, 3))    matriz 3x3 de ceros
Zm = zeros(3, 4)   # np.zeros((3, 4))    matriz 3x4 de ceros
O  = ones(3)       # np.ones((3, 3))     matriz 3x3 de unos
Om = ones(2, 3)    # np.ones((2, 3))     matriz 2x3 de unos
col = zeros(3, 1)  # np.zeros((3, 1))    vector COLUMNA
fila = zeros(1, 3) # np.zeros((1, 3))    vector FILA
Id = I[3]          # np.eye(3)           identidad CUADRADA 3x3
R  = I[3, 4]       # np.eye(3, 4)        identidad RECTANGULAR 3x4
E  = eye(3, 4)     # np.eye(3, 4)        forma explicita, igual que I[3, 4]

print("Z = zeros(3)      forma", Z.shape, "(matriz, no vector)")
print("O = ones(3)       forma", O.shape)
print("Zm = zeros(3, 4)  forma", Zm.shape)
print("Om = ones(2, 3)   forma", Om.shape)
print("zeros(3, 1)       forma", col.shape, "-> columna")
print("zeros(1, 3)       forma", fila.shape, "-> fila")
print("I[3]              forma", Id.shape)
print("I[3, 4]           forma", R.shape, "-> rectangular")
print("\nes lo mismo I[3, 4] que eye(3, 4)?", np.array_equal(R, E))

Z = zeros(3)      forma (3, 3) (matriz, no vector)
O = ones(3)       forma (3, 3)
Zm = zeros(3, 4)  forma (3, 4)
Om = ones(2, 3)   forma (2, 3)
zeros(3, 1)       forma (3, 1) -> columna
zeros(1, 3)       forma (1, 3) -> fila
I[3]              forma (3, 3)
I[3, 4]           forma (3, 4) -> rectangular

es lo mismo I[3, 4] que eye(3, 4)? True


### 1.2.1 Constructores simbólicos

Cuando la matriz va a llevar **letras**, conviene crearla ya simbólica. Para eso
están `zeros_sym`, `ones_sym` y `eye_sym`, que son idénticas a sus versiones
numéricas pero devuelven una matriz de SymPy:

| Numérica | Simbólica (corta) | Simbólica (larga) |
|---|---|---|
| `zeros(3)` | `zeros.sym(3)` | `zeros_sym(3)` |
| `ones(2, 2)` | `ones.sym(2, 2)` | `ones_sym(2, 2)` |
| `eye(3)` | `eye.sym(3)` | `eye_sym(3)` |
| `I[3]` | `I.sym[3]` | — |

La forma corta `zeros.sym(3)` se lee como "los ceros, pero simbólicos".

La ventaja de crearlas simbólicas desde el principio es que imprime los números como si fueran símbolos en vez de flotantes con punto decimal, pero puedes ir llenándolas con números y símbolos mezclados.

Y si ya tienes una matriz numérica, `var(A)` la convierte a simbólica, igual que
`sym(A)` en MATLAB: `var` mira lo que recibe y decide sola si debe **crear
símbolos** (`var('x', 'y')`) o **convertir** (`var(A)`).

Al convertir, los números se vuelven **exactos**, sin punto decimal:

| Antes (flotante) | Después con `var()` |
|---|---|
| `1.0` | `1` |
| `0.1` | `1/10` |
| `7.85` | `157/20` |
| `0.3333333333333333` | `1/3` |
| `1e-18` | `1.00000000000000e-18` (se deja igual, para no destruirlo) |

Así el álgebra simbólica trabaja sin errores de redondeo: `1/3` es exacto,
mientras que `0.3333333333333333` no lo es.

In [ ]:
# Matriz simbolica vacia que se va llenando
A = zeros.sym(2, 2)          # forma corta; zeros_sym(2, 2) es equivalente
print("tipo:", type(A).__name__, "| forma:", A.shape)

x1, q = var('x1', 'q')
A[0, 0] = x1
A[0, 1] = 1
A[1, 0] = 3
A[1, 1] = cos(q)

print("\nA =")
mostrar(A)

print("\nones.sym(2, 3) = \n")
mostrar(ones.sym(2, 3))
print("eye.sym(3) =")
mostrar(eye.sym(3))

# Y los operadores siguen significando lo mismo
print("\nA * A  (elemento a elemento) =")
mostrar(A * A)
print("\nA @ A  (matricial) =")
mostrar(A @ A)

tipo: MatrizRobomat | forma: (2, 2)

A =


<IPython.core.display.Math object>


ones.sym(2, 3) = 



<IPython.core.display.Math object>

eye.sym(3) =


<IPython.core.display.Math object>


A * A  (elemento a elemento) =


<IPython.core.display.Math object>


A @ A  (matricial) =


<IPython.core.display.Math object>

### 1.3 Qué te ahorra la biblioteca

| Si no la tuvieras | Con la biblioteca |
|---|---|
| `import numpy as np` y `import sympy as sym` en cada cuaderno | ya están cargadas |
| `np.array([[1, 2], [3, 4]])` | `M[[1, 2], [3, 4]]` |
| `sym.Matrix([[p, q], [r, s]])` | `M[[p, q], [r, s]]` (detecta las letras) |
| `sym.symbols('p q r s')` | `var('p', 'q', 'r', 's')` |
| `sym.init_printing()` + `display(Math(...))` para ver LaTeX | `mostrar(expresión)` |
| `np.linalg.solve(A, b)` | `solve(A, b)` |
| `np.linalg.det(A)` / `np.linalg.inv(A)` | `det(A)` / `inv(A)` |
| `np.linspace(...)` | `linspace(...)` o el atajo `lin(...)` |
| `plt.plot(...)` | `plot(...)` |

Además, `M[[...]]` evita el error más común del curso: mezclar sin querer un
arreglo numérico con una matriz simbólica. Si escribes un solo símbolo, toda la
matriz se vuelve simbólica y las operaciones siguen funcionando.

> 💡 **No es una biblioteca nueva**: por dentro es NumPy y SymPy. Todo lo que
> aprendas de ellas sigue siendo válido, y puedes usar `np.` y `sym.` cuando
> quieras.

## 2. Operaciones básicas

- Suma y resta: `A + B`, `A - B`.
- Producto matricial: `A @ B` (o `dot(A, B)`).
- Transpuesta: `A.T`.


### 2.1 Los operadores, y cuál usar

Si usaras NumPy y SymPy **por separado**, `*` significaría cosas distintas en
cada una: en NumPy multiplica elemento a elemento, pero en SymPy hace el
producto matricial.

**La biblioteca del curso lo unificó.** Las matrices que salen de `M[[...]]`
tienen la misma semántica de NumPy, sean numéricas o simbólicas:

| Escribes | Resultado | Equivale a |
|---|---|---|
| `A * B` | producto **elemento a elemento** | el `.*` de MATLAB |
| `A @ B` | producto **matricial** | el `*` de MATLAB |
| `A ** 2` | potencia **elemento a elemento** | el `^` de MATLAB, cada entrada al cuadrado |
| `A.T` | traspuesta | cambiar renglones por columnas |

Una sola regla, igual para números y para letras:

> ✅ **Regla:** `@` es producto matricial; `*` es elemento a elemento.

In [ ]:
A = M[[1, 2],
      [3, 4]]

B = M[[2, 0],
      [1, 2]]

print("A =")
print(A)
print("B =")
print(B)

print("\nA + B  (suma, no hay ambiguedad) =")
print(A + B)

print("\nA @ B  (producto MATRICIAL) =")
print(A @ B)
print("     fila 1 de A por columna 1 de B: 1*2 + 2*1 = 4")

print("\nA * B  (producto ELEMENTO A ELEMENTO) =")
print(A * B)
print("     equivale al .* de MATLAB")
print("     cada entrada por su pareja: 1*2 = 2, 2*0 = 0, 3*1 = 3, 4*2 = 8")

print("\nA ** 2  (cada entrada al cuadrado) =")
print(A ** 2)

print("\nA.T  (traspuesta; aqui coincide con NumPy y SymPy) =")
print(A.T)

A =
[[1. 2.]
 [3. 4.]]
B =
[[2. 0.]
 [1. 2.]]

A + B  (suma, no hay ambiguedad) =
[[3. 2.]
 [4. 6.]]

A @ B  (producto MATRICIAL) =
[[ 4.  4.]
 [10.  8.]]
     fila 1 de A por columna 1 de B: 1*2 + 2*1 = 4

A * B  (producto ELEMENTO A ELEMENTO) =
[[2. 0.]
 [3. 8.]]
     equivale al .* de MATLAB
     cada entrada por su pareja: 1*2 = 2, 2*0 = 0, 3*1 = 3, 4*2 = 8

A ** 2  (cada entrada al cuadrado) =
[[ 1.  4.]
 [ 9. 16.]]

A.T  (traspuesta; aqui coincide con NumPy y SymPy) =
[[1. 3.]
 [2. 4.]]


In [ ]:
# Lo mismo con simbolos: el resultado NO cambia
x1, x2 = var('x1', 'x2')
S = M[[x1, 2],
      [3, x2]]
B = var(B)  # convierte B a simbolica, para que el producto sea simbólico

print("S =")
mostrar(S)

print("\nS @ B  (producto MATRICIAL) =")
mostrar(S @ B)
print("\nS * B  (ELEMENTO A ELEMENTO) =")
mostrar(S * B)
print("\nS ** 2 (cada entrada al cuadrado) =")
mostrar(S ** 2)
print("\nEs el MISMO codigo que con matrices numericas.")

S =


<IPython.core.display.Math object>


S @ B  (producto MATRICIAL) =


<IPython.core.display.Math object>


S * B  (ELEMENTO A ELEMENTO) =


<IPython.core.display.Math object>


S ** 2 (cada entrada al cuadrado) =


<IPython.core.display.Math object>


Es el MISMO codigo que con matrices numericas.


### 2.2 ¿Y la traspuesta?

La traspuesta `A.T` **no sustituye** al producto elemento a elemento: son
operaciones distintas que sirven para cosas distintas.

| Operación | Para qué sirve |
|---|---|
| `A * B` (o `hadamard(A, B)`) | multiplicar dos tablas entrada por entrada (escalar columnas, calcular errores relativos, Monte Carlo) |
| `A.T` | darle la vuelta a los renglones y columnas: sirve para $\mathbf{A}^{T}\mathbf{A}$ (matrices de Gram y mínimos cuadrados), para comprobar si una matriz es **simétrica** ($\mathbf{A} = \mathbf{A}^{T}$, frecuente en los casos de la unidad) y para escribir productos punto como `x.T @ y` |

En resumen: `*` es para *combinar datos*, la traspuesta es para *cambiar
la forma* de la matriz. Usa la que corresponda al problema, no una en lugar de la
otra.

In [ ]:
A = M[[1, 2],
      [3, 4]]
B = M[[2, 0],
      [1, 2]]
print("A =")
print(A)
print("B =")
print(B)

print("\nA + B =")
print(A + B)
print("\nA @ B =")
print(A @ B)
print("\nA.T   =")
print(A.T)


A =
[[1. 2.]
 [3. 4.]]
B =
[[2. 0.]
 [1. 2.]]

A + B =
[[3. 2.]
 [4. 6.]]

A @ B =
[[ 4.  4.]
 [10.  8.]]

A.T   =
[[1. 3.]
 [2. 4.]]


## 3. Operaciones de renglón (clave para la eliminación)

La **eliminación Gaussiana** se reduce a una sola operación repetida:

$$R_i \leftarrow R_i - m_{ik}\, R_k$$

Con la sintaxis estilo MATLAB se escribe casi igual que en el pizarrón, solo que empieza desde el elemento 0 en vez de empezar en 1:


In [ ]:
A = M[[ 2,  1, -1],
      [-3, -1,  2],
      [-2,  1,  2]]

# Eliminar el elemento A[1,0] usando el pivote A[0,0]
factor = A[1, 0] / A[0, 0]
A[1, :] = A[1, :] - factor * A[0, :]      
# con [1, :] se indica que se reemplaza 
# todos (:) los elementos de la fila 1,
# mientras que [:, 0] indicaría 
# todos los elementos de la columna 0.

print("A tras eliminar A[1,0]:\n", A)


A tras eliminar A[1,0]:
 [[ 2.   1.  -1. ]
 [ 0.   0.5  0.5]
 [-2.   1.   2. ]]


### 3.1 Esa sintaxis NO es de nuestra biblioteca

Conviene tenerlo claro: `A[1, :]` y `A[:, 0]` son **sintaxis de NumPy y de
SymPy**, no algo que agregue nuestro `robomat.py`. Lo único que aporta la
biblioteca es la forma corta de **crear** la matriz (`M[[...]]`); de ahí en
adelante todo es Python estándar.

La buena noticia es que **funciona igual** en las matrices numéricas y en las
simbólicas:

| Escritura | Significado |
|---|---|
| `A[i, :]` | todos los elementos del **renglón** $i$ |
| `A[:, j]` | todos los elementos de la **columna** $j$ |
| `A[i, j]` | un solo elemento |
| `:` | "desde el principio hasta el final" |

### ⚠️ Trampa: matrices de enteros

Si la matriz se crea con **enteros**, la asignación en el mismo lugar **trunca**
los decimales sin avisar:

```python
A = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]])   # int64
factor = A[1, 0] / A[0, 0]        # -1.5
A[1, :] = A[1, :] - factor * A[0, :]
print(A[1, :])                     # [0 0 0]  ✗  debía ser [0, 0.5, 0.5]
```

El resultado correcto se pierde porque `A` solo puede guardar enteros. **No hay
mensaje de error**: es el peor tipo de fallo.

> ✅ Por eso usamos `M[[...]]`: siempre crea la matriz en punto flotante, así que
> la operación de renglón funciona bien. Y si quieres **exactitud total**, usa
> `var(A)` para pasar a fracciones y no perder nada.

## 4. Matriz aumentada $[\mathbf{A}\mid\mathbf{b}]$

Con `c_[A, b]` concatenas la matriz y el vector en una sola **matriz aumentada**.
Este es el objeto que transformarás durante la eliminación.


In [ ]:
A = M[[3, -0.1, -0.2],
      [0.1,  7, -0.3],
      [0.3, -0.2, 10]]
b = M[[7.85],
      [-19.3],
      [71.4]]

Ab = c_[A, b]
print("Matriz aumentada [A|b] =\n", Ab)


Matriz aumentada [A|b] =
 [[  3.    -0.1   -0.2    7.85]
 [  0.1    7.    -0.3  -19.3 ]
 [  0.3   -0.2   10.    71.4 ]]


### 4.1 Concatenar en horizontal y en vertical

`c_[...]` es de **NumPy** (no de la biblioteca del curso) y une bloques **en
horizontal**, uno al lado del otro. Su pareja es `r_[...]`, que los une **en
vertical**, uno debajo de otro:

| Función | Dirección | Sirve para |
|---|---|---|
| `c_[A, b]` | horizontal ➡️ | matriz aumentada $[\mathbf{A}\mid\mathbf{b}]$ |
| `r_[f1, f2]` | vertical ⬇️ | apilar renglones y armar la matriz fila por fila |

> ✅ **La forma la decide lo que escribes**, igual que en MATLAB:
>
> | Escribes | Obtienes |
> |---|---|
> | `M[[1, 2, 3]]` | **renglón** `(1, 3)` |
> | `M[[1], [2], [3]]` | **columna** `(3, 1)` |
> | `M[[1, 2], [3, 4]]` | matriz `(2, 2)`: cada lista interior es un renglón |
>
> La regla es simple: si dentro hay **números sueltos**, es un renglón; si dentro
> hay **listas**, cada lista es un renglón.

In [ ]:
# Dos renglones y se apilan VERTICALMENTE con r_
f1 = M[[1, 2, 3]]        # numeros sueltos -> forma (1, 3), un renglon
f2 = M[[4, 5, 6]]
print("f1 tiene forma", f1.shape, "-> un renglon")

print("\nr_[f1, f2] (vertical) =\n", r_[f1, f2])

# Dos columnas y se apilan HORIZONTALMENTE con c_
c1 = M[[1], [2], [3]]    # listas -> forma (3, 1), una columna
c2 = M[[4], [5], [6]]
print("\nc1 tiene forma", c1.shape, "-> una columna")
print("\nc_[c1, c2] (horizontal) =\n", c_[c1, c2])

#### Con matrices simbólicas: `apilar_filas` y `apilar_columnas`

`r_` y `c_` son de NumPy y **fallan si hay símbolos** (intentan convertir todo a
número). Para esos casos la biblioteca del curso trae dos funciones que hacen lo
mismo y sí aceptan letras:

| NumPy | Biblioteca del curso | Nombre de MATLAB |
|---|---|---|
| `r_[a, b]` | `apilar_filas(a, b)` | `vertcat(a, b)` |
| `c_[a, b]` | `apilar_columnas(a, b)` | `horzcat(a, b)` |
| — | — | `cat(1, a, b)` / `cat(2, a, b)` |

### 4.2 Concatenar como en MATLAB: `M[[A, b]]`

La forma más cómoda es **meter las matrices dentro de `M[[...]]`**. La biblioteca
detecta que no son números sueltos sino bloques, y los concatena:

| Escribes | Equivale a MATLAB |
|---|---|
| `M[[A, b]]` | `[A, b]` → uno al lado del otro (horizontal) |
| `M[[A], [B]]` | `[A; B]` → uno debajo del otro (vertical) |
| `M[[A, B], [C, D]]` | `[A B; C D]` → matriz por bloques |

#### Por qué esto importa

En Python, `np.array([A, b])` **no** concatena: crea un arreglo de tres
dimensiones o, peor, un arreglo de objetos que contiene matrices como cajas
separadas. El resultado ya no es una matriz y cualquier operación posterior falla
o da basura:

```python
A = np.array([[1., 2.], [3., 4.]])
b = np.array([[5.], [6.]])
np.array([A, b])        # forma (2, 2, 2)  ✗  ¡tres dimensiones!
```

Con `M[[A, b]]` el resultado es la matriz aumentada de verdad, forma `(2, 3)`.

In [ ]:
x = var('x')

# apilar_columnas sirve para la matriz aumentada aunque haya simbolos
S = var([[x, 1],
         [2, x]])
bb = var([[3],
          [4]])
mostrar(apilar_columnas(S, bb))

# apilar_filas arma la matriz renglon por renglon
r1 = var([[x, 2, 3]])
r2 = var([[4, 5, 6]])
mostrar(apilar_filas(r1, r2))

print("\nY con r_ / c_ de NumPy esto daria error por los simbolos.")

In [ ]:
# La forma mas comoda: meter las matrices dentro de M[[...]]
A = M[[1, 2],
      [3, 4]]
b = M[[5],
      [6]]
B = M[[7, 8],
      [9, 10]]

print("Horizontal  M[[A, b]]        forma", M[[A, b]].shape)
print(M[[A, b]])

print("\nVertical    M[[A], [B]]      forma", M[[A], [B]].shape)
print(M[[A], [B]])

print("\nBloques     M[[A, B], [A, B]] forma", M[[A, B], [A, B]].shape)
print(M[[A, B], [A, B]])

print("\nNombres de MATLAB: horzcat(A, b) =", horzcat(A, b).shape,
      "| vertcat(A, B) =", vertcat(A, B).shape)

# Lo que NO hay que hacer: np.array([A, b]) no concatena
print("\nnp.array([A, B]) tiene forma", np.array([A, B]).shape,
      "<- 3 dimensiones, NO es una matriz")
try:
    np.array([A, b])
except Exception as e:
    print("np.array([A, b]) ni siquiera se puede construir:", type(e).__name__)

## 5. Determinante e inversa

- `det(A)`: determinante. Un determinante **cercano a cero** avisa de un sistema mal condicionado.
- `inv(A)`: matriz inversa $\mathbf{A}^{-1}$.


In [ ]:
A = M[[1, 2],
      [3, 4]]

print("det(A)   =", det(A))
print("inv(A)   =\n", inv(A))
print("A @ inv(A) =\n", A @ inv(A))   # debe dar la identidad

## 6. Resolver $\mathbf{A}\mathbf{x}=\mathbf{b}$ con `solve`

`solve(A, b)` es el equivalente a la **barra invertida** de MATLAB `A \ b`.


In [ ]:
A = M[[3, -0.1, -0.2],
      [0.1,  7, -0.3],
      [0.3, -0.2, 10]]
b = M[[7.85],
      [-19.3],
      [71.4]]

x = solve(A, b)
print("x =\n", x)
print("Verificación A @ x =\n", A @ x)   # debe dar b


## 7. ¿Qué significa «resolver»? Interpretación geométrica

Hasta aquí has resuelto sistemas con `solve`, pero no hemos dicho **qué representa
la solución**. Cada ecuación de $\mathbf{A}\mathbf{x}=\mathbf{b}$ es una
**restricción**: el conjunto de todos los puntos que la cumplen.

| Incógnitas | Cada ecuación es... | La solución es... |
|:---:|:---|:---|
| 2 | una **recta** en el plano | el **punto** donde se cruzan las rectas |
| 3 | un **plano** en el espacio | el **punto** donde se cruzan los tres planos |

Las figuras de abajo son **interactivas**: arrástralas con el ratón para girarlas.


In [ ]:
# Dos ecuaciones, dos incógnitas: cada ecuación es una RECTA
try:
    import plotly.graph_objects as go
except ImportError:                     # por si no está instalado
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)
    import plotly.graph_objects as go

x1 = np.linspace(-1.5, 4.5, 200)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x1, y=4 - 2 * x1, name="2x₁ + x₂ = 4",
                         line=dict(width=3, color="#1f77b4")))
fig.add_trace(go.Scatter(x=x1, y=(7 - x1) / 3, name="x₁ + 3x₂ = 7",
                         line=dict(width=3, color="#d62728")))
fig.add_trace(go.Scatter(x=[1], y=[2], mode="markers+text",
                         marker=dict(size=15, color="black"),
                         text=["solución (1, 2)"], textposition="top center",
                         name="solución"))
fig.update_layout(
    title="2 ecuaciones, 2 incógnitas → las rectas se cruzan en UN punto",
    xaxis_title="x₁", yaxis_title="x₂",
    width=720, height=540, hovermode="closest",
    xaxis=dict(range=[-1.5, 4.5], scaleanchor="y", gridcolor="#e8e8e8"),
    yaxis=dict(range=[-1.5, 4.5], gridcolor="#e8e8e8"))
fig.show()


In [ ]:
# Tres ecuaciones, tres incógnitas: cada ecuación es un PLANO.
# Usamos el mismo sistema del Cuaderno 02 / docs, cuya solución es (2, 3, -1).
u = np.linspace(0.4, 3.6, 40)
v = np.linspace(1.4, 4.6, 40)
U, V = np.meshgrid(u, v)

def _solido(color):
    return [[0, color], [1, color]]   # colorscale plana = color uniforme

fig = go.Figure()
fig.add_trace(go.Surface(x=U, y=V, z=2 * U + V - 8, opacity=0.62,
                         colorscale=_solido("#1f77b4"), showscale=False,
                         name="2x₁ + x₂ − x₃ = 8", showlegend=True))
fig.add_trace(go.Surface(x=U, y=V, z=(3 * U + V - 11) / 2, opacity=0.62,
                         colorscale=_solido("#d62728"), showscale=False,
                         name="−3x₁ − x₂ + 2x₃ = −11", showlegend=True))
fig.add_trace(go.Surface(x=U, y=V, z=(2 * U - V - 3) / 2, opacity=0.62,
                         colorscale=_solido("#2ca02c"), showscale=False,
                         name="−2x₁ + x₂ + 2x₃ = −3", showlegend=True))
fig.add_trace(go.Scatter3d(x=[2], y=[3], z=[-1], mode="markers+text",
                           marker=dict(size=7, color="black"),
                           text=["solución (2, 3, −1)"],
                           textposition="top center", name="solución"))
fig.update_layout(
    title="3 ecuaciones, 3 incógnitas → los tres planos se cruzan en UN punto",
    width=820, height=680,
    scene=dict(xaxis_title="x₁", yaxis_title="x₂", zaxis_title="x₃",
               xaxis=dict(range=[0.4, 3.6]),
               yaxis=dict(range=[1.4, 4.6]),
               zaxis=dict(range=[-5, 3])))
fig.show()


### 🔍 ¿Y en los problemas reales?

Los casos de ingeniería de esta unidad tienen **decenas, cientos o incluso miles**
de ecuaciones y de columnas. Ahí ya no se pueden dibujar: con 4 incógnitas la
"figura" viviría en 4 dimensiones, y con 1000 no habría pantalla que la mostrara.

Por eso la representación geométrica sirve para **entender** qué significa
resolver, pero el trabajo real se hace con la matriz $\mathbf{A}$, el vector
$\mathbf{b}$ y el algoritmo. Ese es justo el tema del resto de la unidad y del
[Cuaderno 02](U3_02_Laboratorios_Metodos.ipynb).


## 8. Álgebra simbólica

Con `var()` creas variables simbólicas, y `M[[...]]` detecta automáticamente si la matriz
es simbólica. Es útil para **deducir** ecuaciones (jacobianos, equilibrios, cerchas)
antes de evaluar números.


In [ ]:
q1 = var('q1')

# Vector simbólico y su jacobiano
f = M[[cos(q1)],
      [sin(q1)]]
mostrar(f)

J = jacobiano(f, [q1])
mostrar(J)


In [ ]:
# Evaluar el vector simbólico en 45 grados
valores = evaluar(f, [q1], [rad(45)])
print("Evaluado en 45° =\n", valores)


---
### ✅ ¿Qué sigue?

Ya dominas la sintaxis básica. En el **Cuaderno 02** la usarás para programar los
métodos numéricos de la unidad: **Eliminación Gaussiana**, **Descomposición LU**,
**Gauss-Seidel** y **Thomas/Cholesky**.

📖 Consulta la teoría en [`docs/metodos/`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/tree/main/docs/metodos).
